# **Basic Libraries for CV**

In [ ]:
!pip install -q ultralytics opencv-python numpy

## **Download YOLO model and prediction**

In [9]:
import cv2
from ultralytics import YOLO

# 1. Load the pre-trained YOLO model (using a small nano model for speed)
yolo_model = YOLO("yolov8n.pt")

# 2. Read the source image
image_path = "/content/dog_1.jpeg"  # Replace with your image path
img = cv2.imread(image_path)

# Run inference on the image
results = yolo_model(img)
# 3. Process the results (Iterating over detected objects)


0: 640x448 1 dog, 6.6ms
Speed: 1.5ms preprocess, 6.6ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 448)


In [10]:
yolo_model.export(format="onnx", int8=True)

WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.87 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 223ms
Prepared 4 packages in 1.70s
Installed 4 packages in 247ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to ta

ONNX: export success ✅ 7.5s, saved as 'yolov8n_int8.onnx' (3.4 MB)

Export complete (8.0s)
Results saved to /content/yolov8n_int8.onnx
Predict:         yolo predict task=detect model=yolov8n_int8.onnx imgsz=640 
Validate:        yolo val task=detect model=yolov8n_int8.onnx imgsz=640 data=coco.yaml  
Visualize:       https://netron.app


'yolov8n_int8.onnx'

## **BBOX and cropping**

In [4]:
for result in results:
    boxes = result.boxes  # Box object containing geometric outputs
    print(f"Boxes in the result:{boxes}")

    for box in boxes:
        # ---- Extracting Bounding Box Coordinates ----
        # xyxy format gives: [xmin, ymin, xmax, ymax] as floats
        # We cast to integers to use them as pixel indices
        x_min, y_min, x_max, y_max = map(int, box.xyxy[0])

        # Get class label and confidence score
        class_id = int(box.cls[0])
        label = model.names[class_id]
        confidence = float(box.conf[0])


        print(f"Detected {label} ({confidence:.2f}) at Coordinates: Xmin={x_min}, Ymin={y_min}, Xmax={x_max}, Ymax={y_max}")

        # ---- Geometric Image Cropping ----
        # In computer vision geometry, images are represented as 2D/3D matrices.
        # OpenCV reads images in (Height, Width, Channels) shape.
        # Therefore, to crop using coordinates, we slice the matrix: img[Y_range, X_range]
        cropped_obj = img[y_min:y_max, x_min:x_max]

        # Save or display the geometric crop
        cv2.imwrite(f"cropped_{label}.jpg", cropped_obj)

        # ---- Drawing Bounding Box around the object ----
        # cv2.rectangle takes: (image, top_left_point, bottom_right_point, color_bgr, thickness)
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

        # Add label text just above the bounding box
        text = f"{label} {confidence:.2f}"
        cv2.putText(img, text, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Save the final annotated image with boxes
cv2.imwrite("annotated_output.jpg", img)
print("Processing complete. Images saved.")

Boxes in the result:ultralytics.engine.results.Boxes object with attributes:

cls: tensor([16.], device='cuda:0')
conf: tensor([0.6675], device='cuda:0')
data: tensor([[ 24.6054,  51.6526, 183.0000, 248.3749,   0.6675,  16.0000]], device='cuda:0')
id: None
is_track: False
orig_shape: (275, 183)
shape: torch.Size([1, 6])
xywh: tensor([[103.8027, 150.0137, 158.3946, 196.7224]], device='cuda:0')
xywhn: tensor([[0.5672, 0.5455, 0.8655, 0.7154]], device='cuda:0')
xyxy: tensor([[ 24.6054,  51.6526, 183.0000, 248.3749]], device='cuda:0')
xyxyn: tensor([[0.1345, 0.1878, 1.0000, 0.9032]], device='cuda:0')
Detected dog (0.67) at Coordinates: Xmin=24, Ymin=51, Xmax=183, Ymax=248
Processing complete. Images saved.


## **Masking using SAM**

In [6]:
from ultralytics import SAM
sam_model = SAM("sam2_b.pt")  # or 'sam_b.pt' depending on version

In [7]:
import cv2
from ultralytics import YOLO, SAM
import numpy as np

# 1. Initialize the Models
# YOLOv8n (nano) is used for speed; SAM2 Base is used for balanced precision.
detection_model = YOLO("yolov8n.pt")  # Weights automatically download on first run
sam_model = SAM("sam2_b.pt")           # Weights automatically download on first run

# 2. Load the Image
image_path = "/content/dog_1.jpeg"  # Replace with your image path
img = cv2.imread(image_path)
h, w, c = img.shape

# 3. Stage 1: Run YOLO Detection
# We run inference only on the relevant objects (in this case, just dogs)
# COCO class '16' is 'dog'. This speeds up inference and prevents noise.
detection_results = detection_model(img, classes=[16])

# Collect all detected dog bounding boxes
yolo_boxes = []
for result in detection_results:
    # Ensure any dogs were actually detected
    if len(result.boxes) == 0:
        continue

    # Extract boxes in XYXY format and convert to list
    # shape: [N_dogs, 4]
    boxes_data = result.boxes.xyxy.cpu().numpy().tolist()
    yolo_boxes.extend(boxes_data)

print(f"YOLO found {len(yolo_boxes)} dog(s).")

# 4. Stage 2: Segment the Dogs using YOLO Bboxes as SAM Prompts
# If no dogs found, we exit gracefully
if len(yolo_boxes) == 0:
    print("No dogs detected by YOLO.")
    # Exit or handle as needed
    exit()

# We pass the collected YOLO boxes directly to SAM.
# SAM is prompted specifically to only segment inside those geometric regions.
sam_results = sam_model(img, bboxes=yolo_boxes)

# 5. Process and Visualize SAM Masks
for result in sam_results:
    if result.masks is not None:
        # result.masks contains a list of Mask objects (one for each detected object)
        # Iterate and apply them.
        for i, mask_obj in enumerate(result.masks):
            # Extract binary mask [1, H, W] and convert to CPU numpy array
            mask = mask_obj.data[0].cpu().numpy()

            # 6. Geometric Mask Visualization
            # Scale mask to 0-255 range and convert to uint8
            # A value of 0.5 is typically the segmentation threshold.
            mask_uint8 = (mask > 0.5).astype(np.uint8) * 255

            # Generate a random color for this specific instance mask
            # For visualization, we keep the random color's Alpha channel (transparency).
            color = np.random.randint(0, 255, (3)).tolist()
            colored_mask = np.zeros_like(img, dtype=np.uint8)
            colored_mask[mask_uint8 == 255] = color

            # Generate a translucent overlay using geometric blending
            # (Image + Mask) * Alpha / 2
            img = cv2.addWeighted(img, 1.0, colored_mask, 0.5, 0.0)

            # Draw the original YOLO bounding box over the final visualization
            bbox = yolo_boxes[i]
            x1, y1, x2, y2 = map(int, bbox)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
            cv2.putText(img, "Dog (SAM Mask)", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

# Save the final annotated result
output_path = "yolo_sam_dog_segmentation.jpg"
cv2.imwrite(output_path, img)
print(f"Result saved as: {output_path}")


0: 640x448 1 dog, 9.5ms
Speed: 2.0ms preprocess, 9.5ms inference, 21.1ms postprocess per image at shape (1, 3, 640, 448)
YOLO found 1 dog(s).

0: 1024x1024 1 0, 770.8ms
Speed: 6.5ms preprocess, 770.8ms inference, 6.3ms postprocess per image at shape (1, 3, 1024, 1024)
Result saved as: yolo_sam_dog_segmentation.jpg


In [11]:
!ls -lh yolov8n.pt yolov8n_int8.onnx

-rw-r--r-- 1 root root 3.4M Jul  4 13:48 yolov8n_int8.onnx
-rw-r--r-- 1 root root 6.3M Jul  4 13:23 yolov8n.pt


In [12]:
import os

pt_size = os.path.getsize("yolov8n.pt") / (1024 * 1024)
onnx_size = os.path.getsize("yolov8n_int8.onnx") / (1024 * 1024)

print(f"Original PyTorch Model Size: {pt_size:.2f} MB")
print(f"Quantized INT8 ONNX Model Size: {onnx_size:.2f} MB")

Original PyTorch Model Size: 6.25 MB
Quantized INT8 ONNX Model Size: 3.37 MB
